# Phylodynamics & Pathogen Genomic Surveillance — Real SARS-CoV-2 Time-Scaled Tree

Builds a real ML phylogeny from dated real SARS-CoV-2 genomes, infers a real molecular clock and time-scaled tree, checks real temporal signal, and reconstructs a real skyline effective-population-size trajectory.

## Step 0 — Environment setup

**Real, important trap, already caught:** `pip install treetime` installs a completely unrelated to-do-list app (also hard-imports PyQt6). The real phylodynamics tool is `pip install phylo-treetime` — confirmed working, real CLI subcommands are `homoplasy`, `ancestral`, `mugration`, `clock`, `arg`.

Nextclade and IQ-TREE2 are installed here via **conda/bioconda** rather than hand-downloading GitHub release binaries — bioconda is the real, standard distribution channel for both and sidesteps chasing exact per-version release filenames.

In [1]:
!pip install -q phylo-treetime pandas numpy scipy matplotlib biopython

import os
import json
import subprocess
import pandas as pd
import numpy as np

os.makedirs("bin", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

# real NCBI Datasets CLI -- confirmed current URL from NCBI's own docs
!curl -sL -o bin/datasets "https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets"
!curl -sL -o bin/dataformat "https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/dataformat"
!chmod +x bin/datasets bin/dataformat
os.environ["PATH"] = os.path.abspath("bin") + os.pathsep + os.environ["PATH"]

# real Nextclade + IQ-TREE2 via bioconda -- avoids chasing exact release-binary filenames
!conda install -y -q -c bioconda -c conda-forge nextclade iqtree 2>&1 | tail -20

print(subprocess.run(["datasets", "--version"], capture_output=True, text=True).stdout)
print(subprocess.run(["nextclade", "--version"], capture_output=True, text=True).stdout)
print(subprocess.run(["iqtree", "--version"], capture_output=True, text=True).stdout)
print(subprocess.run(["treetime", "version"], capture_output=True, text=True).stdout)

Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

# All requested packages already installed.

datasets version: 18.40.0

nextclade 3.25.0

IQ-TREE version 3.1.4 for Linux x86 64-bit built Sep 29 2026
Developed by Bui Quang Minh, Thomas Wong, Nhan Ly-Trong, Huaiyan Ren
Contributed by Lam-Tung Nguyen, Dominik Schrempf, Chris Bielow,
Olga Chernomor, Michael Woodhams, Diep Thi Hoang, Heiko Schmidt


treetime 0.12.1



## Step 1 — Real dated SARS-CoV-2 genome subsample, spanning the full real pandemic timeline

**Real, live-discovered redesign:** NCBI's own Datasets v2 API gateway (`api.ncbi.nlm.nih.gov`, POST-based) is unreachable from this real network -- confirmed via direct measurement (a real failed run: 6m43s wall-clock, "giving up after 11 attempt(s)"). Real root cause, also measured directly: a plain GET to the same host completes in ~1 second, so this is a real POST-specific network block (consistent with a firewall/DPI rule that inspects POST bodies), not a general outage. Switched to two real, GET-only data sources instead:

1. **Nextstrain's own real open SARS-CoV-2 metadata** (`data.nextstrain.org/files/ncov/open/metadata.tsv.zst`, confirmed live: 908MB compressed, 9,439,977 real rows) -- streamed directly through `zstd -dc` without ever writing the full decompressed file to disk, used only to pick a real dated subsample. Real confirmed columns used: `host` (value is literally `"Homo sapiens"`, not `"Human"`), `genbank_accession` (missing values are literally `"?"`, not empty), `date`.
2. **NCBI's classic Entrez `efetch` API** (`eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi`), confirmed GET-only, confirmed real params `db=nuccore`, `id=<comma-separated accessions>`, `rettype=fasta`, `retmode=text` -- used only to fetch the actual sequences for the chosen accessions, in batches of ≤200 per NCBI's own real guidance (larger batches should use POST, which we're deliberately avoiding here).

**Real subsampling design, per explicit instruction to cover a larger time period:** instead of one narrow 3-month window, this samples evenly across real calendar quarters spanning the full pandemic (so Alpha, Delta, and multiple Omicron-era lineages are all represented) -- a **date-stratified subsample**, not a plain random one. A plain random sample over the full real span would be dominated by whichever quarter happened to have the most real sequencing effort (recent Omicron-era quarters, since real global sequencing capacity grew hugely over time), which would defeat the point of covering a wide time range. Real, important consequence for later: a single strict molecular clock across this much real evolutionary divergence may show a visibly non-linear root-to-tip relationship -- Step 4 exists specifically to check this before trusting any inferred date, and a messy signal here would itself be a real, correct, teachable finding about rate heterogeneity across variants, not a bug.

Blanks below: stream + filter the real metadata, build a real per-quarter stratified sample, then fetch real sequences from `efetch` in real batches.


In [2]:
import subprocess
import pandas as pd
import numpy as np

# real streaming decompression: pipe curl's real downloaded bytes directly through zstd,
# never writing the full ~908MB-compressed / much-larger-decompressed file to disk
proc = subprocess.Popen(
    ["bash", "-c", "curl -s https://data.nextstrain.org/files/ncov/open/metadata.tsv.zst | zstd -dc"],
    stdout=subprocess.PIPE,
)

USECOLS = ["strain", "genbank_accession", "date", "host", "country", "division"]

# real, live-discovered fix: a single pd.read_csv() on the full ~9.4M-row decompressed stream
# spiked memory hard enough to crash the kernel / WSL connection (measured, not guessed --
# the whole VM going unresponsive, not a Python exception, is the signature of real OOM, the
# same "measure real resource dimensions first" lesson as #41's disk-I/O surprise). Real fix:
# read in bounded chunks and filter each chunk down immediately, so peak memory never exceeds
# one chunk's size instead of the full file.
CHUNK_SIZE = 200_000
filtered_chunks = []
for chunk in pd.read_csv(proc.stdout, sep="\t", usecols=USECOLS, low_memory=False, chunksize=CHUNK_SIZE):
    chunk = chunk[chunk["host"] == "Homo sapiens"]
    chunk = chunk[chunk["genbank_accession"] != "?"]
    chunk = chunk[chunk["date"].str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)]
    filtered_chunks.append(chunk)

proc.wait()
meta_df = pd.concat(filtered_chunks, ignore_index=True)
meta_df["date"] = pd.to_datetime(meta_df["date"])
print("real filtered rows (human host, real open accession, full date precision):", len(meta_df))
print(meta_df.head())


real filtered rows (human host, real open accession, full date precision): 8856762
                strain genbank_accession       date    country   division  \
0    ARG/210316-1/2021          OL989073 2021-03-15  Argentina  Argentina   
1    ARG/210524-2/2021          OL989092 2021-05-17  Argentina  Argentina   
2  ARG/210610-225/2021          OL989093 2021-06-07  Argentina  Argentina   
3   ARG/210613-39/2021          OL989084 2021-06-07  Argentina  Argentina   
4   ARG/210615-23/2021          OL989096 2021-06-14  Argentina  Argentina   

           host  
0  Homo sapiens  
1  Homo sapiens  
2  Homo sapiens  
3  Homo sapiens  
4  Homo sapiens  


In [3]:
# real date-stratified subsample: an even number of real genomes per real calendar quarter,
# so the full pandemic timeline is represented instead of being dominated by recent,
# more heavily-sequenced quarters
meta_df["quarter"] = meta_df["date"].dt.to_period("Q")
N_PER_QUARTER = 15

real_subsample_df = meta_df.groupby("quarter", group_keys=False).apply(
    lambda g: g.sample(n=min(N_PER_QUARTER, len(g)), random_state=42)
)

print("real quarters covered:", real_subsample_df["quarter"].nunique())
print("real total subsample size:", len(real_subsample_df))
print(real_subsample_df["quarter"].value_counts().sort_index())

real_subsample_df[["strain", "genbank_accession", "date"]].to_csv("data/subsample_metadata.csv", index=False)


real quarters covered: 29
real total subsample size: 435
quarter
2019Q4    15
2020Q1    15
2020Q2    15
2020Q3    15
2020Q4    15
2021Q1    15
2021Q2    15
2021Q3    15
2021Q4    15
2022Q1    15
2022Q2    15
2022Q3    15
2022Q4    15
2023Q1    15
2023Q2    15
2023Q3    15
2023Q4    15
2024Q1    15
2024Q2    15
2024Q3    15
2024Q4    15
2025Q1    15
2025Q2    15
2025Q3    15
2025Q4    15
2026Q1    15
2026Q2    15
2026Q3    15
2026Q4    15
Freq: Q-DEC, Name: count, dtype: int64


/tmp/ipykernel_15635/954632906.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  real_subsample_df = meta_df.groupby("quarter", group_keys=False).apply(


In [4]:
# real batched fetch via NCBI's classic Entrez efetch GET API -- confirmed GET-only, so it
# avoids the exact POST block we measured on NCBI's newer Datasets v2 gateway. NCBI's own
# guidance: batches above ~200 UIDs should use POST, so this stays at 150 per real GET call,
# and sleeps briefly to respect the real unauthenticated rate limit of 3 requests/second.
import requests
import time

accessions = real_subsample_df["genbank_accession"].tolist()
BATCH_SIZE = 150
EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

all_fasta_text = []
for i in range(0, len(accessions), BATCH_SIZE):
    batch = accessions[i:i + BATCH_SIZE]
    params = {
        "db": "nuccore",
        "id": ",".join(batch),
        "rettype": "fasta",
        "retmode": "text",
        "tool": "phylodynamics-surveillance-project",
        "email": "yashraj.7156@gmail.com",
    }
    resp = requests.get(EFETCH_URL, params=params, timeout=60)
    resp.raise_for_status()
    all_fasta_text.append(resp.text)
    time.sleep(0.4)

GENOME_FASTA = "data/sars2_subsample.fasta"
with open(GENOME_FASTA, "w") as f:
    f.write("\n".join(all_fasta_text))

print("real sequences fetched:", subprocess.run(["grep", "-c", ">", GENOME_FASTA], capture_output=True, text=True).stdout)


real sequences fetched: 435



## Step 2 — Real alignment, mutation calling, and QC via Nextclade

Real, stable Nextclade dataset name for this pathogen is `sars-cov-2`; real stable CLI output flags are `--output-fasta` (aligned sequences) and `--output-tsv` (per-sequence QC/mutation table).

In [5]:
!nextclade dataset get --name sars-cov-2 --output-dir data/nextclade_dataset

!nextclade run \
    --input-dataset data/nextclade_dataset \
    --output-fasta results/aligned.fasta \
    --output-tsv results/nextclade.tsv \
    "{GENOME_FASTA}"

nextclade_df = pd.read_csv("results/nextclade.tsv", sep="\t")
print(nextclade_df.shape)
print(nextclade_df["qc.overallStatus"].value_counts())

(435, 102)
qc.overallStatus
good        364
mediocre     42
bad          29
Name: count, dtype: int64


In [6]:
# real QC filter -- keep only sequences Nextclade did not flag as bad
qc_pass_df = nextclade_df[nextclade_df["qc.overallStatus"] != "bad"]
qc_pass_names = set(qc_pass_df["seqName"])
print("pre-QC:", len(nextclade_df), "| post-QC:", len(qc_pass_df))

# real, live-discovered ID-matching bug (measured directly, not guessed): Nextclade's
# seqName column keeps the FULL original FASTA header (accession + full description text,
# e.g. "PX060390.1 Severe acute respiratory syndrome coronavirus 2 isolate ..."), while
# Biopython's rec.id truncates to just the first whitespace-delimited token (the accession
# alone, e.g. "PX060390.1") -- a direct set-intersection check confirmed zero real overlap
# between the two as-is. Real fix: extract just the accession token from each qc_pass_names
# entry before comparing.
qc_pass_accessions = {name.split()[0] for name in qc_pass_names}

# real filtered aligned FASTA, QC-passed sequences only
from Bio import SeqIO
records_pass = [rec for rec in SeqIO.parse("results/aligned.fasta", "fasta") if rec.id in qc_pass_accessions]
SeqIO.write(records_pass, "results/aligned_qc_pass.fasta", "fasta")
print("real QC-passed sequences written:", len(records_pass))


pre-QC: 435 | post-QC: 406
real QC-passed sequences written: 406


## Step 3 — Real ML tree via IQ-TREE2

In [7]:
!iqtree -s results/aligned_qc_pass.fasta -m GTR+G -nt AUTO -redo

tree_path = "results/aligned_qc_pass.fasta.treefile"
print("real tree file exists:", os.path.exists(tree_path))

IQ-TREE version 3.1.4 for Linux x86 64-bit built Sep 29 2026
Developed by Bui Quang Minh, Thomas Wong, Nhan Ly-Trong, Huaiyan Ren
Contributed by Lam-Tung Nguyen, Dominik Schrempf, Chris Bielow,
Olga Chernomor, Michael Woodhams, Diep Thi Hoang, Heiko Schmidt

Host:    MAYASTRA (AVX512, FMA3, 7 GB RAM)
Command: iqtree -s results/aligned_qc_pass.fasta -m GTR+G -nt AUTO -redo
Seed:    359169 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Mon Oct 26 21:32:25 2026
Kernel:  AVX+FMA - auto-detect threads (8 CPU cores detected)

Reading alignment file results/aligned_qc_pass.fasta ... Fasta format detected
Reading fasta file: done in 0.214959 secs using 29.55% CPU
Alignment most likely contains DNA/RNA sequences
Constructing alignment: done in 0.123956 secs using 758.5% CPU
Alignment has 406 sequences with 29903 columns, 6306 distinct patterns
890 parsimony-informative, 1540 singleton sites, 27473 constant sites
            Gap/Ambiguity  Composition  p-value
Analyzing seque

## Step 4 — Real root-to-tip regression (check temporal signal BEFORE trusting any date)

In [8]:
# real dates CSV TreeTime needs: 'name' matching real tree tip labels, 'date' in ISO format

# real, third live-discovered bug: the tree's real tip labels carry a GenBank version suffix
# (e.g. "MT291827.1"), confirmed directly via Bio.Phylo, while real_subsample_df's
# genbank_accession column is unversioned (e.g. "MT291827") -- confirmed zero real overlap
# between the two via a direct set-intersection check. Versions aren't all ".1" either, so
# guessing a fixed suffix would be wrong for some records -- the real, general fix: read the
# actual versioned names straight off the real aligned_qc_pass.fasta record IDs (which IQ-TREE
# copied as-is into the tree) and map each bare accession to its real versioned form.
from Bio import SeqIO
real_tree_names = [rec.id for rec in SeqIO.parse("results/aligned_qc_pass.fasta", "fasta")]
version_map = {name.split(".")[0]: name for name in real_tree_names}

dates_for_treetime = real_subsample_df.copy()
dates_for_treetime["name"] = dates_for_treetime["genbank_accession"].map(version_map)
dates_for_treetime = dates_for_treetime.dropna(subset=["name"])
dates_for_treetime["date"] = dates_for_treetime["date"].dt.strftime("%Y-%m-%d")
dates_for_treetime = dates_for_treetime[["name", "date"]]
dates_for_treetime.to_csv("data/dates.csv", index=False)
print(dates_for_treetime.head())

GENOME_LENGTH = 29903  # real, well-established SARS-CoV-2 reference genome length (Wuhan-Hu-1, NC_045512.2)

!treetime clock --tree {tree_path} --dates data/dates.csv --name-column name --date-column date \
    --sequence-length {GENOME_LENGTH} --outdir results/clock --plot-rtt root_to_tip.pdf

print(subprocess.run(["find", "results/clock", "-type", "f"], capture_output=True, text=True).stdout)


               name        date
56178    MT291826.1  2019-12-30
56179    MT291827.1  2019-12-30
8831584  MT019530.1  2019-12-30
2889966  OL549280.1  2019-12-31
56181    MT291829.1  2019-12-30

0.00	-TreeAnc: set-up

Attempting to parse dates...
	Using column 'name' as name. This needs match the taxon names in the tree!!
	Using column 'date' as date.

0.14	TreeTime.reroot: with method or node: least-squares

0.14	TreeTime.reroot: rerooting will ignore covariance and shared ancestry.

1.42	TreeTime.reroot: with method or node: least-squares

1.42	TreeTime.reroot: rerooting will ignore covariance and shared ancestry.

1.76	TreeTime.reroot: with method or node: least-squares

1.76	TreeTime.reroot: rerooting will ignore covariance and shared ancestry.

--- molecular clock model (saved as results/clock/molecular_clock.txt):

Root-Tip-Regression:
 --rate:	7.795e-04
 --r^2:  	0.97

The R^2 value indicates the fraction of variation inroot-to-tip
distance explained by the sampling times.Higher v

In [9]:
# real root-to-tip regression stats are printed to console by the command above and saved in
# results/clock -- read the real clock.txt / molecular_clock.txt TreeTime writes and print it directly
clock_files = [f for f in os.listdir("results/clock") if "clock" in f.lower() or "rate" in f.lower()]
print("real clock-related output files:", clock_files)
for f in clock_files:
    print(f"\n--- {f} ---")
    print(open(os.path.join("results/clock", f)).read())

real clock-related output files: ['molecular_clock.txt']

--- molecular_clock.txt ---
Root-Tip-Regression:
 --rate:	7.795e-04
 --r^2:  	0.97




## Step 5 — Real time-scaled tree + molecular clock

Only proceed here if Step 4's real root-to-tip signal looked reasonable (a visibly positive, roughly-linear relationship in `root_to_tip.pdf` and a real, sensible clock rate printed above — SARS-CoV-2's real published clock rate is on the order of 8e-4 substitutions/site/year).

In [10]:
!treetime --tree {tree_path} --dates data/dates.csv --name-column name --date-column date \
    --sequence-length {GENOME_LENGTH} --outdir results/timetree --confidence

print(subprocess.run(["find", "results/timetree", "-type", "f"], capture_output=True, text=True).stdout)



Attempting to parse dates...
	Using column 'name' as name. This needs match the taxon names in the tree!!
	Using column 'date' as date.

0.00	-TreeAnc: set-up
Outside of covariation aware mode TreeTime cannot estimate confidence
intervals without specified standard deviation of the clock
rate.Please specify '--clock-std-dev' or rerun with '--covariation'.
Will proceed without confidence estimation

0.09	TreeTime.reroot: with method or node: least-squares

0.09	TreeTime.reroot: rerooting will ignore covariance and shared ancestry.

0.84	TreeTime.reroot: with method or node: least-squares

0.84	TreeTime.reroot: rerooting will ignore covariance and shared ancestry.

1.59	###TreeTime.run: INITIAL ROUND

20.67	TreeTime.reroot: with method or node: least-squares

20.67	TreeTime.reroot: rerooting will ignore covariance and shared ancestry.

21.06	###TreeTime.run: rerunning timetree after rerooting

41.20	###TreeTime.run: ITERATION 1 out of 2 iterations

41.20	DEPRECATION WARNING. TreeTime.re

## Step 6 — Real skyline effective population size over time

In [11]:
!treetime --tree {tree_path} --dates data/dates.csv --name-column name --date-column date \
    --sequence-length {GENOME_LENGTH} --coalescent skyline --n-skyline 20 --outdir results/skyline

print(subprocess.run(["find", "results/skyline", "-type", "f"], capture_output=True, text=True).stdout)



Attempting to parse dates...
	Using column 'name' as name. This needs match the taxon names in the tree!!
	Using column 'date' as date.

0.00	-TreeAnc: set-up

0.09	TreeTime.reroot: with method or node: least-squares

0.10	TreeTime.reroot: rerooting will ignore covariance and shared ancestry.

0.71	TreeTime.reroot: with method or node: least-squares

0.71	TreeTime.reroot: rerooting will ignore covariance and shared ancestry.

1.30	###TreeTime.run: INITIAL ROUND

26.88	TreeTime.reroot: with method or node: least-squares

26.88	TreeTime.reroot: rerooting will ignore covariance and shared ancestry.

27.43	###TreeTime.run: rerunning timetree after rerooting

50.39	###TreeTime.run: ITERATION 1 out of 2 iterations

51.61	DEPRECATION WARNING. TreeTime.resolve_polytomies: You are resolving
     	polytomies using the old 'greedy' mode. This is not well suited for large
     	polytomies. Stochastic resolution will become the default in future
     	versions. To switch now, rerun with the flag `

## Step 7 — Export real dated-node table

In [12]:
# real TreeTime output: dates.tsv in the timetree outdir has real per-node inferred calendar dates
dated_nodes_path = "results/timetree/dates.tsv"
print("real dated-nodes file exists:", os.path.exists(dated_nodes_path))

if os.path.exists(dated_nodes_path):
    dated_nodes_df = pd.read_csv(dated_nodes_path, sep="\t")
    dated_nodes_df.to_csv("results/dated_nodes.csv", index=False)
    print(dated_nodes_df.head())
else:
    # real fallback: list the real timetree outdir and adjust the filename above to match what TreeTime
    # actually wrote for this version -- filenames have shifted slightly across real TreeTime releases
    print(subprocess.run(["find", "results/timetree", "-type", "f"], capture_output=True, text=True).stdout)

print("\nreal results directory:", os.listdir("results"))

real dated-nodes file exists: True
          #node        date  numeric date
0  NODE_0000404  2019-07-09   2019.520124
1  NODE_0000110  2019-10-13   2019.781527
2  NODE_0000111  2020-11-03   2020.839786
3    OU029185.1  2021-02-24   2021.149315
4  NODE_0000112  2020-11-04   2020.844119

real results directory: ['aligned.fasta', 'aligned_qc_pass.fasta', 'aligned_qc_pass.fasta.bionj', 'aligned_qc_pass.fasta.ckp.gz', 'aligned_qc_pass.fasta.iqtree', 'aligned_qc_pass.fasta.log', 'aligned_qc_pass.fasta.mldist', 'aligned_qc_pass.fasta.treefile', 'clock', 'dated_nodes.csv', 'nextclade.tsv', 'skyline', 'timetree']
